**OBSOLETE -- superseded by the 2026-08-25 reorientation plan.**

Already served its purpose: the CSV subset it exports is in place under
`data/raw/` and in use. Kept as a historical record of how that export
was done, not maintained going forward. See README.md for the current
plan.

# 00 - Exportar un subconjunto local (gold) desde Kaggle

Objetivo: NO bajar el dataset completo (~0.5 TB, no cabe en disco local -
ver README.md). En su lugar, correr esto **en un Kaggle Notebook**
contra el dataset montado, para copiar a `/kaggle/working/` (1) los 4
CSVs de metadatos completos (`train.csv`, `test.csv`, `train_series.csv`,
`test_series.csv` - son solo texto/tablas, pesan poco aunque cubran los
4,407 estudios) y (2) los DICOM de solo los 58 estudios **gold**
(`train_series/<StudyInstanceUID>/...`), suficientes para prototipar
`src/data.py`/`src/model.py` y para los tests, sin acercarse al tamano
del dataset completo.

Al terminar, descarga el zip generado desde el panel "Output" de este
kernel de Kaggle (boton de descarga, no el CLI) y extraelo en
`data/raw/` del repo local, replicando la misma estructura:
`data/raw/train.csv`, `data/raw/train_series.csv`,
`data/raw/train_series/<StudyInstanceUID>/<SeriesInstanceUID>/*.dcm`,
etc. `src/config.py::DATA_RAW_DIR` ya apunta ahi cuando no se detecta
Kaggle.

In [ ]:
import shutil
from pathlib import Path

import pandas as pd

RAW_DIR = Path("/kaggle/input/competitions/rsna-knee-abnormality-detection")
assert RAW_DIR.exists(), f"Dataset no montado en {RAW_DIR} - anadelo como input del kernel."

OUT_DIR = Path("/kaggle/working/data_raw")
OUT_DIR.mkdir(parents=True, exist_ok=True)

# Copiado de src/config.py::OFFICIAL_LABEL_COLUMNS.
OFFICIAL_LABEL_COLUMNS = {
    "acl_injury": "ACL",
    "mcl_injury": "MCL",
    "medial_meniscus_tear": "Medial Meniscus",
    "lateral_meniscus_tear": "Lateral Meniscus",
    "oa_medial_compartment": "Medial OA",
    "oa_lateral_compartment": "Lateral OA",
    "oa_patellofemoral_compartment": "PF OA",
    "effusion": "Effusion",
    "synovitis": "Synovitis",
    "bakers_cyst": "Baker's",
    "bone_contusion": "Contusion",
    "fracture": "Fracture",
}
LABEL_COLS = list(OFFICIAL_LABEL_COLUMNS.values())

## 1. Identificar el subset gold (58 estudios)

In [ ]:
train = pd.read_csv(RAW_DIR / "train.csv")
n_labels_present = train[LABEL_COLS].notna().sum(axis=1)
gold_mask = n_labels_present == len(LABEL_COLS)
gold_study_ids = set(train.loc[gold_mask, "StudyInstanceUID"])
print(f"Estudios gold identificados: {len(gold_study_ids)}")

## 2. Copiar los 4 CSVs completos (pequenos, cubren todo train/test)

Estos no dependen del subset: `Report` en `train.csv` vive para las
4,407 filas y no pesa como imagen, asi que copiarlo entero deja la Fase
2 (EDA de informes) corriendo 100% en local despues de este paso, sin
tocar Kaggle de nuevo.

In [ ]:
for name in ["train.csv", "test.csv", "train_series.csv", "test_series.csv"]:
    src = RAW_DIR / name
    if src.exists():
        shutil.copy2(src, OUT_DIR / name)
        print(f"copiado: {name} ({src.stat().st_size / 1e6:.1f} MB)")
    else:
        print(f"AVISO: {name} no existe en {RAW_DIR}, se omite")

## 3. Medir el tamano de los DICOM gold ANTES de copiar

No copiar a ciegas - los 58 estudios pueden pesar mas de lo esperado
segun resolucion/n de cortes. Medir primero, decidir despues (mismo
principio que `N_STUDIES` en `01_eda_dicom.ipynb`: no recorrer/copiar
el arbol completo sin saber cuanto cuesta).

In [ ]:
def dir_size_bytes(path):
    return sum(f.stat().st_size for f in path.rglob("*") if f.is_file())

gold_dirs = [RAW_DIR / "train_series" / sid for sid in gold_study_ids]
gold_dirs = [d for d in gold_dirs if d.exists()]
print(f"Carpetas de estudio gold encontradas: {len(gold_dirs)} / {len(gold_study_ids)}")

total_bytes = sum(dir_size_bytes(d) for d in gold_dirs)
print(f"Tamano total de los DICOM gold: {total_bytes / 1e9:.2f} GB")
print("Revisar este numero contra el espacio libre en disco local antes de la celda 4.")

## 4. Copiar los DICOM gold

Poner `PROCEED_WITH_DICOM_COPY = True` solo despues de revisar el
tamano impreso arriba (celda 3). Si solo hace falta la Fase 2 (EDA de
informes), esta celda y la siguiente se pueden saltar por completo -
los CSVs de la celda 2 ya son suficientes.

In [ ]:
PROCEED_WITH_DICOM_COPY = False  # cambiar a True tras revisar el tamano de la celda 3

if PROCEED_WITH_DICOM_COPY:
    dest_series_dir = OUT_DIR / "train_series"
    dest_series_dir.mkdir(exist_ok=True)
    for d in gold_dirs:
        dest = dest_series_dir / d.name
        if not dest.exists():
            shutil.copytree(d, dest)
    print(f"Copiados {len(gold_dirs)} estudios gold a {dest_series_dir}")
else:
    print("PROCEED_WITH_DICOM_COPY=False - solo se exportan los CSVs (celda 2).")

## 5. Empaquetar en un solo zip para descargar

Un solo archivo es mas simple de bajar desde el panel Output de Kaggle
que decenas de miles de archivos sueltos.

In [ ]:
zip_path = shutil.make_archive("/kaggle/working/rsna_knee_local_subset", "zip", OUT_DIR)
print(f"Zip listo: {zip_path} ({Path(zip_path).stat().st_size / 1e6:.1f} MB)")
print("\nSiguiente paso (fuera de este notebook): descargar este zip desde el panel")
print("Output/Data de Kaggle y extraerlo en data/raw/ del repo local, de forma que")
print("quede data/raw/train.csv, data/raw/train_series.csv, etc.")